# **FUDMASTUDENTBUDDY BOT**
## How May I Help You?

---

## **PROJECT OWNER DETAILS**

| Detail | Information |
|--------|-------------|
| **Name** | Sanusi Shafii |
| **Matriculation Number** | CSA\2023\27683 |
| **Email Address** | s.shafii27683@fudutsinma.edu.ng.com |
| **Phone Number** | +2349125006811 |
| **Department** | Computer Science and Information Technology |
| **Faculty** | Computing |
| **University** | Federal University Dutsin-Ma |
| **Location** | Katsina State, Nigeria |

---

## **PROJECT INFORMATION**

**Project Title:** Development of FUDMA Student Chatbot

**Project Duration:** November 2023 - November 2024

In [10]:
import json
import numpy as np
import pickle
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import random

import nltk
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("Libraries imported successfully!")

# ============================================================================
# CELL 3: COMPREHENSIVE INTENTS DATABASE
# ============================================================================

INTENTS_DATA = {
    "intents": [
        # Greetings and Basic Interaction
        {
            "tag": "greeting",
            "patterns": [
                "Hello", "Hi", "Hey", "Good morning", "Good afternoon",
                "Good evening", "Greetings", "What's up", "Howdy"
            ],
            "responses": [
                "Hello! I am the FUDMA Study Assistant. How can I help you today?",
                "Good day! Welcome to FUDMA Study Assistant. What would you like to know?",
                "Hello! I'm here to assist with your academic queries at FUDMA. How may I help?"
            ]
        },
        {
            "tag": "goodbye",
            "patterns": [
                "Bye", "Goodbye", "See you later", "Talk to you later",
                "I have to go", "Thanks bye", "Catch you later"
            ],
            "responses": [
                "Goodbye! Best of luck with your studies at FUDMA.",
                "See you later! Feel free to return anytime you need assistance.",
                "Take care! Good luck with your academic pursuits."
            ]
        },
        {
            "tag": "thanks",
            "patterns": [
                "Thanks", "Thank you", "That's helpful", "Thanks a lot",
                "I appreciate it", "Cheers", "Thank you very much"
            ],
            "responses": [
                "You're welcome! Happy to help.",
                "Glad I could assist! Let me know if you need anything else.",
                "My pleasure! Feel free to ask more questions."
            ]
        },

        # University Information
        {
            "tag": "fudma_location",
            "patterns": [
                "Where is FUDMA", "FUDMA location", "Where is the university",
                "FUDMA address", "How to get to FUDMA", "Where is Federal University Dutsin-Ma"
            ],
            "responses": [
                "Federal University Dutsin-Ma (FUDMA) is located in Dutsin-Ma, Katsina State, Nigeria.",
                "FUDMA is situated in Dutsin-Ma Local Government Area of Katsina State, Northwestern Nigeria."
            ]
        },
        {
            "tag": "fudma_history",
            "patterns": [
                "When was FUDMA established", "FUDMA history", "Tell me about FUDMA",
                "FUDMA founding", "When did FUDMA start", "Background of FUDMA"
            ],
            "responses": [
                "Federal University Dutsin-Ma was established in February 2011 as one of the new generation federal universities in Nigeria.",
                "FUDMA was founded in 2011 by the Federal Government of Nigeria to expand access to university education in the northern region."
            ]
        },
        {
            "tag": "fudma_faculties",
            "patterns": [
                "What faculties are in FUDMA", "List FUDMA faculties", "Faculties at FUDMA",
                "Schools in FUDMA", "Academic divisions at FUDMA", "What can I study at FUDMA"
            ],
            "responses": [
                "FUDMA has several faculties including: Faculty of Science, Faculty of Agriculture, Faculty of Education, Faculty of Humanities and Social Sciences, and Faculty of Computing.",
                "The university comprises faculties such as Science, Agriculture, Education, Humanities and Social Sciences, and Computing, offering diverse academic programs."
            ]
        },
        {
            "tag": "fudma_courses",
            "patterns": [
                "What courses does FUDMA offer", "Available programs at FUDMA",
                "Degree programs at FUDMA", "What can I study", "List of courses"
            ],
            "responses": [
                "FUDMA offers undergraduate and postgraduate programs in sciences, agriculture, education, humanities, social sciences, and computing. Visit the university website or contact the admissions office for the complete list of available programs.",
                "The university provides various degree programs across multiple disciplines including Computer Science, Mathematics, Biology, Chemistry, Economics, Education, and Agricultural Sciences among others."
            ]
        },

        # Admissions
        {
            "tag": "admission_requirements",
            "patterns": [
                "What are admission requirements", "How to gain admission",
                "FUDMA admission process", "Requirements for admission",
                "What do I need for admission", "Admission criteria"
            ],
            "responses": [
                "Admission requirements include: minimum of 5 O'Level credits including English and Mathematics, acceptable JAMB score, and passing the FUDMA post-UTME screening. Specific requirements vary by program.",
                "To gain admission to FUDMA, you need O'Level results (5 credits including English and Mathematics), JAMB UTME score meeting the departmental cutoff, and successful performance in the post-UTME examination."
            ]
        },
        {
            "tag": "jamb_cutoff",
            "patterns": [
                "What is FUDMA cutoff mark", "JAMB cutoff for FUDMA",
                "Minimum JAMB score", "Cutoff mark", "Required JAMB score"
            ],
            "responses": [
                "FUDMA's general cutoff mark is typically 180 and above, but competitive departments may have higher cutoff marks. Check the official admissions page for specific departmental cutoffs.",
                "The university sets a minimum JAMB score of 180, though admission is competitive and departments may require higher scores depending on the number of applicants."
            ]
        },
        {
            "tag": "application_process",
            "patterns": [
                "How to apply to FUDMA", "Application process",
                "How do I apply", "Steps to apply", "Application procedure"
            ],
            "responses": [
                "To apply: 1) Write JAMB UTME and select FUDMA as first choice, 2) Register for post-UTME on the FUDMA portal when announced, 3) Pay the application fee, 4) Complete and submit the application form online.",
                "The application process involves choosing FUDMA in JAMB, monitoring the university website for post-UTME registration dates, purchasing the application form online, and completing the screening exercise."
            ]
        },

        # Registration and Academic Procedures
        {
            "tag": "course_registration",
            "patterns": [
                "How to register courses", "Course registration process",
                "How do I register for courses", "Steps for course registration",
                "Register for semester courses"
            ],
            "responses": [
                "Course registration is done online through the FUDMA student portal. Log in with your credentials, select courses according to your level and semester, ensure prerequisites are met, and submit before the deadline.",
                "To register courses: 1) Access the student portal, 2) Navigate to course registration, 3) Select appropriate courses for your level, 4) Verify your selection, 5) Submit and print your course form."
            ]
        },
        {
            "tag": "add_drop_courses",
            "patterns": [
                "Can I add or drop courses", "How to drop a course",
                "Add/drop courses", "Change registered courses", "Modify course registration"
            ],
            "responses": [
                "There is usually an add/drop period at the beginning of each semester (typically first two weeks). You can modify your course registration through the student portal during this period.",
                "Course addition or withdrawal is permitted during the add/drop period announced by the university. Access the portal and make changes before the deadline to avoid penalties."
            ]
        },
        {
            "tag": "credit_load",
            "patterns": [
                "How many units can I register", "Maximum credit load",
                "Minimum units to register", "Credit unit limit", "Course load"
            ],
            "responses": [
                "Typically, students register between 15-24 credit units per semester. Minimum is usually 15 units for full-time status, maximum is 24 units. Students with excellent CGPA may be allowed to register extra units.",
                "The standard credit load is 15-24 units per semester. First-year students often have fixed course loads while higher levels have some flexibility within the allowed range."
            ]
        },

        # Examinations
        {
            "tag": "exam_eligibility",
            "patterns": [
                "Am I eligible for exams", "Exam eligibility requirements",
                "Can I write exams", "Requirements to sit for exams", "Exam qualification"
            ],
            "responses": [
                "To be eligible for examinations, you must: 1) Have properly registered the course, 2) Meet minimum attendance requirement (usually 75%), 3) Have no outstanding fees, 4) Have submitted required assignments.",
                "Exam eligibility requires proper course registration, at least 75% attendance, payment of all fees, and completion of continuous assessments. Failure to meet these may result in disqualification."
            ]
        },
        {
            "tag": "exam_timetable",
            "patterns": [
                "When are exams", "Exam timetable", "Examination schedule",
                "When will exams hold", "Exam dates", "Check exam schedule"
            ],
            "responses": [
                "Examination timetables are published on the university portal and notice boards at least two weeks before exams begin. Check your faculty notice board and the student portal regularly.",
                "The exam schedule is typically released 2-3 weeks before the examination period. Monitor the university website and your department's notice board for the official timetable."
            ]
        },
        {
            "tag": "exam_rules",
            "patterns": [
                "What are exam rules", "Examination regulations",
                "Exam conduct", "What to bring to exam", "Exam guidelines"
            ],
            "responses": [
                "Exam rules include: arrive 15 minutes early, bring student ID and exam permit, no phones or electronic devices, no malpractice, follow invigilator instructions, write only with blue or black pen.",
                "Key examination regulations: maintain silence, no cheating or assistance, submit answer booklets before leaving, electronic devices are prohibited, and any violation may lead to disciplinary action."
            ]
        },

        # Results and CGPA
        {
            "tag": "check_results",
            "patterns": [
                "How to check results", "Where to see my results",
                "Check exam results", "View my grades", "Access results"
            ],
            "responses": [
                "Results are released on the student portal. Log in with your credentials and navigate to the results section to view your semester grades and CGPA.",
                "To check results: 1) Visit the FUDMA student portal, 2) Log in with your matriculation number and password, 3) Click on 'Results' to view your academic performance."
            ]
        },
        {
            "tag": "cgpa_calculation",
            "patterns": [
                "How is CGPA calculated", "Calculate CGPA", "CGPA computation",
                "How to calculate my CGPA", "GPA calculation method"
            ],
            "responses": [
                "CGPA is calculated as: Total Grade Points earned divided by Total Credit Units registered. Grade Points = Grade Point Value (A=5, B=4, C=3, D=2, E=1, F=0) multiplied by Credit Units.",
                "To calculate CGPA: multiply each course's credit unit by its grade point value (A=5, B=4, C=3, D=2, E=1, F=0), sum all grade points, then divide by total credit units registered."
            ]
        },
        {
            "tag": "grading_system",
            "patterns": [
                "What is the grading system", "FUDMA grading scale",
                "Grade points", "Grading criteria", "Score to grade conversion"
            ],
            "responses": [
                "FUDMA grading system: A (70-100) = 5 points, B (60-69) = 4 points, C (50-59) = 3 points, D (45-49) = 2 points, E (40-44) = 1 point, F (0-39) = 0 points (Fail).",
                "The grading scale is: A (Excellent: 70-100), B (Very Good: 60-69), C (Good: 50-59), D (Fair: 45-49), E (Pass: 40-44), F (Fail: below 40)."
            ]
        },

        # Academic Support
        {
            "tag": "study_tips",
            "patterns": [
                "How to study effectively", "Study tips", "How can I study better",
                "Improve my grades", "Study techniques", "Academic success tips"
            ],
            "responses": [
                "Effective study tips: 1) Attend all lectures, 2) Take comprehensive notes, 3) Form study groups, 4) Review materials regularly, 5) Practice past questions, 6) Seek clarification from lecturers during office hours, 7) Start exam preparation early.",
                "To improve academically: maintain consistent attendance, actively participate in class, create a study schedule, use multiple resources (textbooks, online materials), teach concepts to others, and prioritize difficult subjects."
            ]
        },
        {
            "tag": "library_resources",
            "patterns": [
                "Library hours", "Access library", "Library resources",
                "Use the library", "Library services", "Study at the library"
            ],
            "responses": [
                "The FUDMA library provides access to books, journals, online databases, and study spaces. Operating hours are typically Monday-Friday (8am-10pm) and Saturdays (9am-6pm). Bring your student ID for access.",
                "Library services include book lending, reference materials, computer access, internet facilities, and quiet study areas. Register with your student ID to borrow books and access digital resources."
            ]
        },
        {
            "tag": "office_hours",
            "patterns": [
                "Lecturer office hours", "When can I see my lecturer",
                "Consultation hours", "Meet with professor", "Talk to my teacher"
            ],
            "responses": [
                "Lecturers maintain office hours for student consultations, typically posted on department notice boards or announced in class. You can also email lecturers to schedule appointments outside regular office hours.",
                "Each lecturer has designated consultation hours, usually 2-3 hours per week. Check your course outline, departmental notice board, or ask during lectures for specific times."
            ]
        },

        # Fees and Financial Information
        {
            "tag": "school_fees",
            "patterns": [
                "How much is school fees", "Tuition fees", "Cost of schooling",
                "School fees amount", "What do I pay", "Fee structure"
            ],
            "responses": [
                "School fees vary by faculty and level. Generally, fees include tuition, library, laboratory, sports, and administrative charges. Contact the bursary department or check the university website for current fee schedules.",
                "Fee amounts differ across programs. For specific information on current tuition and other charges for your faculty and level, visit the bursary department or check official notices on the student portal."
            ]
        },
        {
            "tag": "payment_deadline",
            "patterns": [
                "When to pay fees", "Fee payment deadline", "Last date to pay fees",
                "Payment due date", "When is fee deadline"
            ],
            "responses": [
                "Fee payment deadlines are announced each semester, usually before or during registration period. Late payment may incur penalties. Check the student portal and notice boards for specific deadline dates.",
                "Payment deadlines are published at the start of each semester. Ensure you pay before the deadline to avoid late payment charges and registration complications."
            ]
        },
        {
            "tag": "payment_methods",
            "patterns": [
                "How to pay school fees", "Payment methods", "Pay fees online",
                "Where to pay fees", "Fee payment process"
            ],
            "responses": [
                "School fees can be paid through: 1) Online payment via the student portal using Remita, 2) Bank deposits to designated university accounts, 3) Payment at designated bank branches. Always print your receipt after payment.",
                "To pay fees: log into the student portal, generate a Remita Retrieval Reference (RRR), pay at any bank or online, then upload payment evidence on the portal for confirmation."
            ]
        },

        # Campus Life
        {
            "tag": "accommodation",
            "patterns": [
                "Student accommodation", "Hostel facilities", "Where can I stay",
                "Campus housing", "How to get hostel", "Dormitory"
            ],
            "responses": [
                "FUDMA provides on-campus hostel accommodation on a competitive basis. Priority is often given to first-year and final-year students. Apply through the student portal during the hostel allocation period.",
                "Hostel accommodation is available but limited. Applications open at specific periods announced by the university. Off-campus housing is also available in Dutsin-Ma town."
            ]
        },
        {
            "tag": "student_union",
            "patterns": [
                "Student union", "Join student associations", "Student government",
                "Student activities", "Campus organizations", "Clubs at FUDMA"
            ],
            "responses": [
                "FUDMA has an active Student Union Government (SUG) and various departmental associations, clubs, and societies including religious groups, sports clubs, and academic associations. Join during orientation or registration periods.",
                "Students can participate in numerous organizations: SUG, faculty associations, departmental bodies, sports clubs, religious groups, and special interest societies. These enhance your university experience."
            ]
        },
        {
            "tag": "health_services",
            "patterns": [
                "Medical services", "Health center", "Campus clinic",
                "Where to get medical help", "Student health", "Sick bay"
            ],
            "responses": [
                "FUDMA has a health center providing basic medical services to students. Services include consultations, first aid, and referrals to hospitals for serious cases. Present your student ID for access.",
                "The university health center offers medical consultations, emergency services, and health education. For emergencies, contact campus security or go directly to the health center."
            ]
        },

        # Academic Calendar
        {
            "tag": "semester_dates",
            "patterns": [
                "When does semester start", "Academic calendar", "Semester dates",
                "School resumption", "When does school resume", "Session timeline"
            ],
            "responses": [
                "The academic calendar with semester start and end dates, examination periods, and holidays is published on the university website and student portal at the beginning of each session.",
                "Semester dates and the full academic calendar are released before the session begins. Check the official university website or student portal for the current session's calendar."
            ]
        },
        {
            "tag": "holidays",
            "patterns": [
                "School holidays", "Break periods", "When is holiday",
                "Mid-semester break", "Public holidays", "Vacation"
            ],
            "responses": [
                "The university observes public holidays and has scheduled breaks including mid-semester breaks and end-of-semester vacations. Specific dates are in the academic calendar.",
                "Holiday periods include mid-semester breaks, semester breaks, and public holidays. Refer to the academic calendar for exact dates and duration of breaks."
            ]
        },

        # Postgraduate Information
        {
            "tag": "postgraduate_programs",
            "patterns": [
                "Postgraduate programs", "Masters degree", "PhD programs",
                "Graduate school", "Postgraduate admission", "PG programs at FUDMA"
            ],
            "responses": [
                "FUDMA offers postgraduate programs including Masters and PhD degrees in various disciplines. Contact the School of Postgraduate Studies for admission requirements, available programs, and application procedures.",
                "The university runs postgraduate programs at Masters and Doctoral levels. Requirements include a good bachelor's degree, relevant experience (for some programs), and success in the entrance examination."
            ]
        },

        # Student Services
        {
            "tag": "id_card",
            "patterns": [
                "Student ID card", "Get my ID card", "Matriculation card",
                "How to collect ID", "Student identification"
            ],
            "responses": [
                "Student ID cards are processed by the ICT unit. Usually collected during or after registration. You will need passport photographs and proof of registration. Check with your department or ICT unit for collection details.",
                "To obtain your student ID card: ensure you have completed registration, submitted required photographs, and paid necessary fees. Collection dates are announced by the university administration."
            ]
        },
        {
            "tag": "transcript",
            "patterns": [
                "How to get transcript", "Request transcript", "Academic transcript",
                "Apply for transcript", "Transcript process"
            ],
            "responses": [
                "To request a transcript: 1) Complete the transcript application form, 2) Pay the required fee, 3) Submit to the registry, 4) Wait for processing (usually 2-4 weeks). You can collect it in person or have it sent directly to institutions.",
                "Transcript requests are handled by the University Registry. Complete the application form, make payment, submit required documents, and allow processing time. Urgent processing may be available at additional cost."
            ]
        },

        # Help and Support
        {
            "tag": "portal_issues",
            "patterns": [
                "Cannot access portal", "Portal problems", "Forgot password",
                "Portal login issues", "Reset portal password", "Portal not working"
            ],
            "responses": [
                "For portal access issues: 1) Ensure you are using the correct matriculation number and password, 2) Clear browser cache and try again, 3) Use the password reset option if forgotten, 4) Contact ICT support if problems persist.",
                "Portal login problems can often be resolved by: clearing cookies and cache, trying a different browser, using the password recovery feature, or contacting the ICT helpdesk during working hours."
            ]
        },
        {
            "tag": "complaints",
            "patterns": [
                "Lodge complaint", "Who do I report to", "Make a complaint",
                "Report an issue", "Complain about", "File a grievance"
            ],
            "responses": [
                "Complaints can be directed to: 1) Your course adviser for academic issues, 2) HOD for departmental matters, 3) Dean of Student Affairs for welfare issues, 4) SUG for student-related concerns. Formal complaints should be written.",
                "To lodge a complaint: start with your immediate supervisor (course adviser or HOD), escalate to higher authorities if unresolved. The Student Affairs office handles welfare and disciplinary matters."
            ]
        },

        # Contact Information
        {
            "tag": "contact_university",
            "patterns": [
                "How to contact FUDMA", "University contact", "FUDMA phone number",
                "Email FUDMA", "Contact information", "Reach the university"
            ],
            "responses": [
                "You can contact FUDMA through: Official website: www.fudma.edu.ng, General inquiries: Check website for phone numbers, Email: info@fudma.edu.ng (verify current email from official sources), Physical address: Federal University Dutsin-Ma, PMB 5001, Dutsin-Ma, Katsina State.",
                "For inquiries, visit the administrative blocks on campus during working hours (8am-4pm Monday-Friday), call the university through numbers on the official website, or send emails to relevant departments."
            ]
        },

        # Default and Error Handling
        {
            "tag": "unclear",
            "patterns": [
                "I don't understand", "What do you mean", "Explain better",
                "Can you clarify", "I'm confused"
            ],
            "responses": [
                "I apologize for any confusion. Could you please rephrase your question or provide more details about what you need help with?",
                "Let me try to help you better. Can you provide more specific information about your query?"
            ]
        },
        {
            "tag": "out_of_scope",
            "patterns": [
                "Tell me a joke", "What is your name", "How old are you",
                "Sing a song", "Tell me about yourself"
            ],
            "responses": [
                "I am the FUDMA Study Assistant, designed to help with academic and university-related queries. Please ask me questions about FUDMA, admissions, courses, examinations, or student services.",
                "I specialize in providing information about FUDMA and academic matters. How can I assist you with your studies or university-related questions?"
            ]
        }
    ]
}

# Save intents to file
with open('fudma_intents.json', 'w') as f:
    json.dump(INTENTS_DATA, f, indent=2)

print(f"Intents database created with {len(INTENTS_DATA['intents'])} intents!")

# ============================================================================
# CELL 4: TEXT PREPROCESSING CLASS
# ============================================================================

class TextPreprocessor:
    """Handle text preprocessing for NLP tasks"""

    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))
        # Keep important question words
        self.stop_words -= {'what', 'when', 'where', 'who', 'how', 'why', 'which'}

    def clean_text(self, text):
        """Clean and normalize text"""
        # Convert to lowercase
        text = text.lower()
        # Remove special characters but keep alphanumeric and spaces
        text = re.sub(r'[^a-z0-9\s]', '', text)
        # Remove extra whitespace
        text = ' '.join(text.split())
        return text

    def tokenize(self, text):
        """Tokenize text into words"""
        return word_tokenize(text)

    def remove_stopwords(self, tokens):
        """Remove stopwords from token list"""
        return [token for token in tokens if token not in self.stop_words]

    def lemmatize(self, tokens):
        """Lemmatize tokens"""
        return [self.lemmatizer.lemmatize(token) for token in tokens]

    def preprocess(self, text):
        """Complete preprocessing pipeline"""
        text = self.clean_text(text)
        tokens = self.tokenize(text)
        tokens = self.remove_stopwords(tokens)
        tokens = self.lemmatize(tokens)
        return ' '.join(tokens)

preprocessor = TextPreprocessor()
print("Text preprocessor initialized!")

# ============================================================================
# CELL 5: CHATBOT ENGINE CLASS
# ============================================================================

class FUDMAStudyAssistant:
    """Main chatbot engine using TF-IDF and cosine similarity"""

    def __init__(self, intents_file='fudma_intents.json'):
        self.intents = self.load_intents(intents_file)
        self.preprocessor = TextPreprocessor()
        self.vectorizer = TfidfVectorizer(
            max_features=1000,
            ngram_range=(1, 2),  # Include bigrams for better context
            min_df=1
        )
        self.prepare_model()

    def load_intents(self, filename):
        """Load intents from JSON file"""
        with open(filename, 'r') as f:
            data = json.load(f)
        return data['intents']

    def prepare_model(self):
        """Prepare TF-IDF model with all patterns"""
        self.patterns = []
        self.pattern_to_intent = {}

        # Collect all patterns and map them to intents
        for intent in self.intents:
            for pattern in intent['patterns']:
                processed_pattern = self.preprocessor.preprocess(pattern)
                self.patterns.append(processed_pattern)
                self.pattern_to_intent[processed_pattern] = intent['tag']

        # Fit vectorizer on all patterns
        self.pattern_vectors = self.vectorizer.fit_transform(self.patterns)
        print(f"Model trained on {len(self.patterns)} patterns across {len(self.intents)} intents")

    def get_intent(self, user_input, threshold=0.3):
        """Identify intent from user input using cosine similarity"""
        # Preprocess user input
        processed_input = self.preprocessor.preprocess(user_input)

        # Vectorize input
        input_vector = self.vectorizer.transform([processed_input])

        # Calculate cosine similarities
        similarities = cosine_similarity(input_vector, self.pattern_vectors)[0]

        # Get best match
        max_similarity = np.max(similarities)

        if max_similarity < threshold:
            return None, max_similarity

        # Get the pattern with highest similarity
        best_match_idx = np.argmax(similarities)
        best_pattern = self.patterns[best_match_idx]
        intent_tag = self.pattern_to_intent[best_pattern]

        return intent_tag, max_similarity

    def get_response(self, user_input):
        """Get chatbot response for user input"""
        intent_tag, confidence = self.get_intent(user_input)

        if intent_tag is None:
            return "I'm not sure I understand your question. Could you please rephrase it or ask about FUDMA admissions, courses, exams, fees, or campus life?"

        # Find intent and return random response
        for intent in self.intents:
            if intent['tag'] == intent_tag:
                response = random.choice(intent['responses'])
                return response

        return "I'm having trouble understanding. Please ask about FUDMA-related topics."

    def chat(self):
        """Run command-line chat interface"""
        print("\n" + "="*70)
        print("FUDMA STUDY ASSISTANT")
        print("Federal University Dutsin-Ma - Academic Support Chatbot")
        print("="*70)
        print("\nHello! I'm here to help with your FUDMA academic queries.")
        print("Ask me about admissions, courses, exams, fees, campus life, etc.")
        print("Type 'quit' or 'exit' to end the conversation.\n")

        while True:
            user_input = input("You: ").strip()

            if not user_input:
                continue

            if user_input.lower() in ['quit', 'exit', 'bye', 'goodbye']:
                print("\nAssistant: Goodbye! Best of luck with your studies at FUDMA.")
                break

            response = self.get_response(user_input)
            print(f"Assistant: {response}\n")

    def save_model(self, filename='fudma_chatbot_model.pkl'):
        """Save trained model"""
        model_data = {
            'vectorizer': self.vectorizer,
            'patterns': self.patterns,
            'pattern_to_intent': self.pattern_to_intent,
            'pattern_vectors': self.pattern_vectors
        }
        with open(filename, 'wb') as f:
            pickle.dump(model_data, f)
        print(f"Model saved to {filename}")

# Initialize chatbot
chatbot = FUDMAStudyAssistant()
print("FUDMA Study Assistant initialized and ready!")

# ============================================================================
# CELL 6: TEST THE CHATBOT
# ============================================================================

# Test questions covering various intents
test_questions = [
    "Hello",
    "Where is FUDMA located?",
    "What are the admission requirements?",
    "How do I register for courses?",
    "When are the exams?",
    "How is CGPA calculated?",
    "What is the grading system?",
    "How much are the school fees?",
    "Tell me about the library",
    "How can I get accommodation?",
    "What postgraduate programs are available?",
    "How do I check my results?",
    "What faculties does FUDMA have?",
    "Can I drop a course?",
    "How to study effectively?",
    "Thanks for your help"
]

print("\n" + "="*70)
print("TESTING CHATBOT WITH SAMPLE QUESTIONS")
print("="*70 + "\n")

correct_responses = 0
for i, question in enumerate(test_questions, 1):
    print(f"{i}. Question: {question}")
    response = chatbot.get_response(question)
    intent_tag, confidence = chatbot.get_intent(question)
    print(f"   Intent: {intent_tag} (Confidence: {confidence:.2f})")
    print(f"   Response: {response}")
    print()
    if intent_tag is not None:
        correct_responses += 1

accuracy = (correct_responses / len(test_questions)) * 100
print(f"Intent Recognition Accuracy: {accuracy:.1f}%")
print("="*70)

# ============================================================================
# CELL 7: COMMAND-LINE INTERFACE
# ============================================================================

def run_cli():
    """Run the command-line interface"""
    chatbot.chat()

# Uncomment the line below to start CLI mode
# run_cli()

print("\nTo start command-line chat, run: run_cli()")

# ============================================================================
# CELL 8: GRADIO WEB INTERFACE
# ============================================================================

import gradio as gr

def gradio_chat(message, history):
    """Gradio chat function"""
    response = chatbot.get_response(message)
    return response

# Create Gradio interface
def create_gradio_interface():
    """Create and launch Gradio web interface"""

    # Custom CSS for professional look
    custom_css = """
    .gradio-container {
        font-family: 'Arial', sans-serif;
    }
    .chat-message {
        padding: 10px;
        margin: 5px 0;
    }
    """

    # Create the interface
    demo = gr.ChatInterface(
        fn=gradio_chat,
        title="🎓 FUDMA Study Assistant",
        description="""
        **Federal University Dutsin-Ma - Academic Support Chatbot**

        Ask me about:
        - Admissions and Requirements
        - Course Registration
        - Examinations and Grading
        - School Fees and Payments
        - Campus Life and Facilities
        - Academic Procedures
        - Student Services

        *Simple cademic assistance for FUDMA students*
                 *Develop by Yasir Abdulhakeeb*
                 *Department of Computer Science and IT*
                 *For Enquiry: 08129388105*
        """,
        examples=[
            "What are the admission requirements?",
            "How do I register for courses?",
            "How is CGPA calculated?",
            "What is the JAMB cutoff mark?",
            "Tell me about FUDMA faculties",
            "How can I check my results?",
            "What are the exam rules?",
            "How do I get student accommodation?"
        ],
        theme=gr.themes.Soft(),
        css=custom_css
    )

    return demo

# Create interface instance
gradio_interface = create_gradio_interface()

print("\nGradio interface created!")
print("To launch web interface, run: gradio_interface.launch()")

# ============================================================================
# CELL 9: LAUNCH WEB INTERFACE
# ============================================================================

# Launch Gradio interface (uncomment to start)
# gradio_interface.launch(share=True)

print("\n" + "="*70)
print("SETUP COMPLETE!")
print("="*70)
print("\nYou have two options to interact with the chatbot:")
print("\n1. COMMAND-LINE INTERFACE:")
print("   Run: run_cli()")
print("\n2. WEB INTERFACE (Recommended):")
print("   Run: gradio_interface.launch(share=True)")
print("\n" + "="*70)

# ============================================================================
# CELL 10: EVALUATION AND METRICS
# ============================================================================

def evaluate_chatbot():
    """Comprehensive evaluation of chatbot performance"""

    print("\n" + "="*70)
    print("CHATBOT EVALUATION REPORT")
    print("="*70 + "\n")

    # Test dataset with expected intents
    eval_data = [
        ("Hello there", "greeting"),
        ("Where is the university located", "fudma_location"),
        ("What do I need for admission", "admission_requirements"),
        ("How to register courses", "course_registration"),
        ("When are exams held", "exam_timetable"),
        ("Calculate my CGPA", "cgpa_calculation"),
        ("School fees payment", "school_fees"),
        ("Library opening hours", "library_resources"),
        ("Student hostel", "accommodation"),
        ("Thanks a lot", "thanks"),
        ("What is FUDMA", "fudma_history"),
        ("Available courses", "fudma_courses"),
        ("Postgraduate programs", "postgraduate_programs"),
        ("How to check results", "check_results"),
        ("Exam eligibility", "exam_eligibility")
    ]

    correct = 0
    total = len(eval_data)

    print("Testing intent recognition:\n")
    for question, expected_intent in eval_data:
        predicted_intent, confidence = chatbot.get_intent(question)
        is_correct = predicted_intent == expected_intent

        if is_correct:
            correct += 1
            status = "✓"
        else:
            status = "✗"

        print(f"{status} Q: {question}")
        print(f"  Expected: {expected_intent} | Got: {predicted_intent} | Confidence: {confidence:.2f}\n")

    accuracy = (correct / total) * 100

    print("="*70)
    print(f"RESULTS: {correct}/{total} correct")
    print(f"ACCURACY: {accuracy:.1f}%")
    print("="*70)

    # Model statistics
    print(f"\nMODEL STATISTICS:")
    print(f"- Total Intents: {len(chatbot.intents)}")
    print(f"- Total Patterns: {len(chatbot.patterns)}")
    print(f"- Vocabulary Size: {len(chatbot.vectorizer.vocabulary_)}")
    print(f"- Feature Dimensions: {chatbot.pattern_vectors.shape[1]}")

    return accuracy

# Run evaluation
evaluation_accuracy = evaluate_chatbot()

# ============================================================================
# CELL 11: SAVE MODEL
# ============================================================================

# Save the trained model
chatbot.save_model('fudma_chatbot_model.pkl')

# Save intents separately for reference
with open('fudma_intents_backup.json', 'w') as f:
    json.dump(INTENTS_DATA, f, indent=2)

print("\n✓ Model and data saved successfully!")
print("\nFiles created:")
print("- fudma_chatbot_model.pkl (trained model)")
print("- fudma_intents.json (intents database)")
print("- fudma_intents_backup.json (backup)")

# ============================================================================
# CELL 12: QUICK START GUIDE
# ============================================================================

print("\n" + "="*70)
print("QUICK START GUIDE")
print("="*70)
print("""
STEP 1: All dependencies are installed and model is trained ✓

STEP 2: Choose your interface

Option A - Web Interface (Recommended):
    gradio_interface.launch(share=True)

    This will give you:
    - Beautiful web interface
    - Public URL to share with others
    - Easy to use chat interface
    - Mobile friendly

Option B - Command Line Interface:
    run_cli()

    This will give you:
    - Terminal-based chat
    - Quick testing
    - Lightweight option

STEP 3: Test the chatbot with questions like:
    - "What are the admission requirements?"
    - "How do I register for courses?"
    - "How is CGPA calculated?"
    - "What faculties does FUDMA have?"
    - "Tell me about school fees"

PERFORMANCE METRICS:
    ✓ Intent Recognition Accuracy: """ + f"{evaluation_accuracy:.1f}%" + """
    ✓ Total Intents Supported: """ + f"{len(chatbot.intents)}" + """
    ✓ Response Database: """ + f"{sum(len(intent['responses']) for intent in chatbot.intents)} responses" + """

READY TO START!
Run one of the commands above to begin chatting.
""")
print("="*70)


Libraries imported successfully!
Intents database created with 38 intents!
Text preprocessor initialized!
Model trained on 215 patterns across 38 intents
FUDMA Study Assistant initialized and ready!

TESTING CHATBOT WITH SAMPLE QUESTIONS

1. Question: Hello
   Intent: greeting (Confidence: 1.00)
   Response: Good day! Welcome to FUDMA Study Assistant. What would you like to know?

2. Question: Where is FUDMA located?
   Intent: fudma_location (Confidence: 1.00)
   Response: FUDMA is situated in Dutsin-Ma Local Government Area of Katsina State, Northwestern Nigeria.

3. Question: What are the admission requirements?
   Intent: admission_requirements (Confidence: 1.00)
   Response: Admission requirements include: minimum of 5 O'Level credits including English and Mathematics, acceptable JAMB score, and passing the FUDMA post-UTME screening. Specific requirements vary by program.

4. Question: How do I register for courses?
   Intent: course_registration (Confidence: 1.00)
   Response: To

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


In [11]:
gradio_interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5a52247d0e93545050.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
